# Module 5 — Entity Resolution and the Promotion Path

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

Module 4 populated the LGD (Lexical Graph Database) with data from three sources.
But the same real-world customer may appear in Pattern A (customer master) and
Pattern B (transactions) with different identifiers. The customer master says
"Customer C-001, Jordan Rivera." The transaction system says "Account holder
JRivera-7742." Are these the same person?

**Entity Resolution (ER)** is the process of determining that two or more records
across different sources refer to the same real-world entity. It is one of the
hardest problems in data integration — and one of the most consequential for FSI,
where getting it wrong means either missing a wealth signal (false negative) or
contacting the wrong person (false positive).

This module teaches you to:

- Understand what entity resolution is and why it matters for cross-source graphs
- Configure AWS Entity Resolution with rule-based matching (primary) and
  ML-based matching (fallback for low-confidence cases)
- Write a promotion script that moves resolved entities from the LGD to the SLGD
  with full PROV-O (W3C Provenance Ontology) attribution
- Understand why promotion is a governed, discrete action — not an automatic pipeline

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **Entity Resolution (ER)** | The process of determining that two or more records from different sources refer to the same real-world entity. Also called "record linkage," "deduplication," or "identity resolution." |
| **AWS Entity Resolution** | An AWS service that matches records across data sources using configurable matching rules or machine learning. It produces a mapping: "these source records all refer to the same entity." |
| **Rule-based matching** | Matching records using deterministic rules you define: "if first_name matches AND last_name matches AND zip code matches, they are the same person." Always produces the same result. Component class: DETERMINISTIC. |
| **ML-based matching** | Matching records using a trained model that scores similarity. Non-deterministic — the model may produce different confidence scores across versions. Component class: PROBABILISTIC-EXPLAINABLE. |
| **Promotion** | The governed action of moving validated data from the LGD (raw, unvalidated) to the SLGD (curated, authoritative). Not automatic — requires an explicit, auditable action. |
| **PROV-O (W3C Provenance Ontology)** | A W3C standard for recording who did what to what and when. In ATLAS, every promoted entity carries PROV-O attribution: where it came from, what process produced it, and when. |
| **prov:wasDerivedFrom** | A PROV-O property that links a derived entity to its source. In ATLAS: "this SLGD entity was derived from these LGD triples." |
| **prov:wasGeneratedBy** | A PROV-O property that links an entity to the activity that produced it. In ATLAS: "this SLGD entity was generated by this Entity Resolution workflow run." |
| **Confidence score** | A number between 0 and 1 indicating how certain the ER system is that two records refer to the same entity. Rule-based matches have confidence 1.0 (certain). ML matches have confidence < 1.0. |
| **LGD (Lexical Graph Database)** | The raw-data Neptune cluster. Holds unvalidated triples from source systems. Nothing here is used for compliance decisions. |
| **SLGD (Semantic Layer Graph Database)** | The curated Neptune cluster. Holds validated, FIBO-aligned data with provenance. This is what applications query. |

## Why promotion is governed, not automatic

This is a critical architectural decision. The spec is explicit:

> "The promotion script is intentionally not automated as a continuous pipeline.
> It is a discrete, observable action with a log entry. Model risk management
> cannot defend a continuous, opaque promotion of probabilistic outputs into
> the system of record."

Think of it like a bank's wire transfer approval: the system prepares the transfer,
but a human (or a governed process with an audit trail) approves it. The LGD is
the preparation; promotion is the approval; the SLGD is the system of record.

## Prerequisites

- Module 4 complete (LGD populated with data from all three patterns)
- The CloudFormation stack `atlas-neptune-twotier` running

## Deliverables

- A populated SLGD with resolved customer entities, each with PROV-O metadata
- `ontology/extensions/prov-o-bindings.ttl` — PROV-O vocabulary for the promotion path
- A promotion log showing entity counts, matching methods, and confidence distribution

## Architecture class for this module

**MIXED.** Rule-based entity resolution is DETERMINISTIC (same inputs, same outputs).
ML-based entity resolution is PROBABILISTIC-EXPLAINABLE (non-deterministic but
produces a confidence score per match). The promotion script itself is DETERMINISTIC
(same ER mapping + same LGD data = same SLGD output).

In [ ]:
# Workshop dependency setup — installs into THIS kernel's Python interpreter.
# Using sys.executable guarantees we install into the kernel, not the terminal Python.
# Safe to re-run: pip skips packages that are already installed at the correct version.
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    '--disable-pip-version-check',
    '-r', 'shared/requirements.txt'
])
print('Dependencies ready.')

## How This Connects to Competency Questions

In Module 1, you wrote Competency Questions (CQs) — the testable questions your
ontology must answer. Module 5 is where those questions start getting answered
with *real data* rather than a minimal test graph.

When you promote entities from the LGD (Lexical Graph Database) to the SLGD
(Semantic Layer Graph Database), the promoted data must be able to answer the
same Competency Questions. For example:

- CQ1 ("Which customers have generated a wealth signal?") requires promoted
  Customer entities to exist in the SLGD
- CQ3 ("Which household relationships does this customer have?") requires the
  household membership links to survive promotion
- CQ7 ("What specific observations were used?") requires transaction data to
  be traceable through the promotion path

The WealthSignal derivation at the end of this module is the first time the
architecture *computes* answers to Competency Questions from real data — signals
are derived from promoted transactions, not loaded from a file. This is the
"computed, not loaded" principle in action: CQs are answered by running queries
against derived data.

In [ ]:
import sys
sys.path.insert(0, '../notebooks/shared')

import json
import boto3
from datetime import datetime
from pathlib import Path
import atlas_synthetic
import atlas_sparql

print('ATLAS Module 5 — Entity Resolution and the Promotion Path')
print(f'Synthetic data seed: {atlas_synthetic.ATLAS_SEED}')

# Neptune endpoints from Module 3
cfn = boto3.client('cloudformation', region_name='us-east-1')
try:
    stack = cfn.describe_stacks(StackName='atlas-neptune-twotier')['Stacks'][0]
    outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}
    LGD_ENDPOINT = outputs['LGDEndpoint']
    SLGD_ENDPOINT = outputs['SLGDEndpoint']
    print(f'\nNeptune LGD:  {LGD_ENDPOINT}:8182')
    print(f'Neptune SLGD: {SLGD_ENDPOINT}:8182')
except Exception as e:
    print(f'Could not retrieve Neptune endpoints: {e}')
    LGD_ENDPOINT = ''
    SLGD_ENDPOINT = ''

## What Entity Resolution Actually Does

### The problem

Your LGD has data from three sources. The same customer — Jordan Rivera — appears as:

| Source | Identifier | Name | Evidence |
|--------|-----------|------|----------|
| Pattern A (customer master) | `C-001` | Jordan Rivera | household_id, segment, state |
| Pattern B (transactions) | `account-7742` | J. Rivera | transaction amounts, dates |
| Pattern C (events) | `event-customer-001` | (no name) | signal type, amount |

Are these the same person? A human can probably tell. But with 200 customers and
thousands of transactions, you need a systematic process.

### How AWS Entity Resolution works

1. You define a **schema mapping** — which columns from each source correspond
   to which matching attributes (name, address, account number, etc.)
2. You define **matching rules** — either deterministic rules ("if X matches AND
   Y matches, they are the same") or ML-based ("score the similarity and accept
   above threshold 0.85")
3. You run the **matching workflow** — it produces a mapping table:
   `{source_record_id} → {resolved_entity_id}`
4. You use the mapping to **merge** records that refer to the same entity

### Workshop path vs Production path

| Workshop | Production |
|----------|------------|
| Synthetic data with known matches (same customer_id across sources) | Real data where matches are unknown |
| Rule-based matching on customer_id (deterministic, confidence 1.0) | Rule-based + ML fallback (mixed confidence) |
| Promotion script run manually in notebook | Promotion as a Step Functions workflow with approval gate |
| 200 entities resolved | Millions of entities resolved in batches |

The workshop uses a simplified ER approach (matching on customer_id which is
shared across our synthetic sources) to focus on the **promotion path** and
**PROV-O attribution** — which are the architecturally important parts.

In [ ]:
# Simulate Entity Resolution
# In production, this would be an AWS Entity Resolution workflow.
# For the workshop, we match on customer_id (which is shared across our synthetic sources).

customers = atlas_synthetic.generate_customers(n=200)
accounts = atlas_synthetic.generate_accounts(customers)
transactions = atlas_synthetic.generate_transactions(accounts, lookback_days=90)

# Build the ER mapping: customer_id -> resolved entity
# In production, AWS Entity Resolution would produce this mapping.
er_mapping = {}
for c in customers:
    cid = c['customer_id']
    er_mapping[cid] = {
        'resolved_entity_id': cid,  # In our synthetic data, IDs already match
        'matching_method': 'RULE_BASED',
        'confidence': 1.0,
        'matched_sources': ['customer_master', 'transaction_history'],
        'matching_rule': 'customer_id exact match',
    }

# Simulate some ML-based matches (lower confidence) for demonstration
import random
rng = random.Random(atlas_synthetic.ATLAS_SEED + 10)
ml_match_count = 15  # 15 customers resolved via ML (simulated)

for i, cid in enumerate(list(er_mapping.keys())[:ml_match_count]):
    er_mapping[cid]['matching_method'] = 'ML_BASED'
    er_mapping[cid]['confidence'] = round(rng.uniform(0.75, 0.98), 3)

print('Entity Resolution Results')
print('=' * 60)
print(f'Total entities resolved: {len(er_mapping)}')

rule_based = [e for e in er_mapping.values() if e['matching_method'] == 'RULE_BASED']
ml_based = [e for e in er_mapping.values() if e['matching_method'] == 'ML_BASED']

print(f'  Rule-based matches: {len(rule_based)} (confidence: 1.0)')
print(f'  ML-based matches:   {len(ml_based)} (confidence: {min(e["confidence"] for e in ml_based):.3f} - {max(e["confidence"] for e in ml_based):.3f})')
print()
print('Confidence distribution (ML matches):')
for e in sorted(ml_based, key=lambda x: x['confidence'])[:5]:
    print(f'  {e["resolved_entity_id"][:12]}... confidence: {e["confidence"]}')

## The Promotion Path with PROV-O Attribution

### What PROV-O records

Every entity promoted from the LGD to the SLGD carries three pieces of provenance:

1. **prov:wasDerivedFrom** — which LGD source triple(s) this entity came from
2. **prov:wasGeneratedBy** — which Entity Resolution workflow run produced the match
3. **atlas:confidence** — how certain the ER system is about the match

In Turtle, a promoted Customer looks like this:

```turtle
# The promoted entity in the SLGD
atlas-inst:customer-C001-resolved
    a atlas:Customer ;
    atlas:customerId "C-001" ;
    atlas:promotedFrom atlas-lgd:customer-C001 ;      # LGD source
    atlas:promotedBy atlas-inst:er-run-2026-05-08 ;   # ER workflow run
    atlas:confidence "1.0"^^xsd:decimal ;             # Match confidence
    atlas:matchingMethod "RULE_BASED" .               # How it was matched

# The promotion activity
atlas-inst:er-run-2026-05-08
    a atlas:PromotionActivity ;
    atlas:promotionTimestamp "2026-05-08T21:00:00Z"^^xsd:dateTime ;
    atlas:promotionOperator "workshop-participant" .
```

### Why this matters for MRM (Model Risk Management)

When a regulator asks "how did this customer end up in your wealth-signal system?",
the answer is a SPARQL query:

```sparql
SELECT ?source ?method ?confidence ?timestamp WHERE {
    ?customer atlas:customerId "C-001" ;
              atlas:promotedFrom ?source ;
              atlas:promotedBy ?activity ;
              atlas:confidence ?confidence ;
              atlas:matchingMethod ?method .
    ?activity atlas:promotionTimestamp ?timestamp .
}
```

The answer is: "This customer was promoted from LGD source X, via rule-based
matching with confidence 1.0, on this date, by this operator." That is the
audit trail MRM requires.

## Outputs Are Computed, Not Loaded

A critical architectural principle in ATLAS: **the system produces signals from data
the customer already has.** No external data is imported to create wealth signals.
No pre-computed signal instances are loaded from a file.

This matters for three reasons:

1. **Regulatory defensibility.** When a regulator asks "where did this signal come
   from?", the answer must trace back to in-bank observations — transactions,
   balances, account events — that the institution already holds. If the signal
   came from an external feed or a pre-loaded file, the provenance chain breaks.

2. **Reproducibility.** Given the same source data and the same derivation logic,
   the system must produce the same signals. Pre-loaded outputs cannot be
   reproduced because their derivation is opaque.

3. **Freshness.** Signals must reflect the current state of the customer's data.
   A pre-loaded signal from last month is stale. A derived signal is always
   current because it is computed from the latest promoted data.

### What this means concretely

- **WealthSignal instances** are derived by SPARQL CONSTRUCT queries that pattern-match
  against promoted customer data (transactions above threshold, household balances
  above threshold, etc.). They are never loaded from a JSON file.

- **Score instances** are produced by the SageMaker XGBoost endpoint at inference time.
  They are never pre-computed and stored.

- **RoutingDecision instances** are produced by the Step Functions state machine
  when a signal is processed. They are never pre-assigned.

The only pre-loaded data in ATLAS is **source data** (customer master, transaction
history, advisory relationships) and **reference data** (the ontology, SKOS codelists,
SHACL shapes). Everything else is computed at runtime.

### The old pattern (what we replaced)

Earlier iterations of this workshop shipped an `event-stream.json` file containing
pre-generated wealth-eligibility events. This violated the "computed, not loaded"
principle — the events existed before the derivation logic ran. The current
architecture derives signals from the transaction data that Pattern B loads into
the LGD, using the SPARQL CONSTRUCT queries shown later in this module.

In [ ]:
# The Promotion Script
# This is the governed action that moves resolved entities from LGD to SLGD.
# It is intentionally NOT automated. Each run is a discrete, logged event.

import requests

ATLAS_NS = 'https://github.com/your-org/atlas/ontology#'
INST_NS = 'https://github.com/your-org/atlas/instance#'
PROV_NS = 'http://www.w3.org/ns/prov#'
XSD_NS = 'http://www.w3.org/2001/XMLSchema#'
RDF_TYPE = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'

def sparql_update_slgd(update_query):
    try:
        url = f'https://{SLGD_ENDPOINT}:8182/sparql'
        resp = requests.post(url, data={'update': update_query},
                            headers={'Content-Type': 'application/x-www-form-urlencoded'},
                            verify=True, timeout=30)   # Neptune uses an Amazon-issued (RDS CA) certificate trusted by the default certifi bundle; no custom CA handling needed
        return resp.status_code == 200
    except:
        return False

def sparql_query_slgd(query):
    try:
        url = f'https://{SLGD_ENDPOINT}:8182/sparql'
        resp = requests.post(url, data={'query': query},
                            headers={'Accept': 'application/sparql-results+json'},
                            verify=True, timeout=30)   # Neptune uses an Amazon-issued (RDS CA) certificate trusted by the default certifi bundle; no custom CA handling needed
        return resp.json() if resp.status_code == 200 else None
    except:
        return None

# Test SLGD connectivity
neptune_available = sparql_query_slgd('SELECT (1 AS ?test) WHERE {}') is not None

# Promotion run metadata
promotion_run_id = f'promotion-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
promotion_timestamp = datetime.utcnow().isoformat() + 'Z'

print('Promotion Path Execution')
print('=' * 60)
print(f'Run ID:    {promotion_run_id}')
print(f'Timestamp: {promotion_timestamp}')
print(f'Operator:  workshop-participant')
print(f'Neptune:   {"reachable" if neptune_available else "not reachable (generating triples only)"}')
print()

# Generate promotion triples with PROV-O attribution
promotion_triples = []

# The promotion activity itself
act_uri = f'<{INST_NS}{promotion_run_id}>'
promotion_triples.append(f'{act_uri} <{RDF_TYPE}> <{ATLAS_NS}PromotionActivity> .')
promotion_triples.append(f'{act_uri} <{ATLAS_NS}promotionTimestamp> "{promotion_timestamp}"^^<{XSD_NS}dateTime> .')
promotion_triples.append(f'{act_uri} <{ATLAS_NS}promotionOperator> "workshop-participant"^^<{XSD_NS}string> .')

# Promote each resolved entity
promoted_count = 0
for cid, er_result in er_mapping.items():
    entity_uri = f'<{INST_NS}customer-{cid}-resolved>'
    lgd_source = f'<{INST_NS}customer-{cid}>'
    
    promotion_triples.append(f'{entity_uri} <{RDF_TYPE}> <{ATLAS_NS}Customer> .')
    promotion_triples.append(f'{entity_uri} <{ATLAS_NS}customerId> "{cid}"^^<{XSD_NS}string> .')
    promotion_triples.append(f'{entity_uri} <{ATLAS_NS}promotedFrom> {lgd_source} .')
    promotion_triples.append(f'{entity_uri} <{ATLAS_NS}promotedBy> {act_uri} .')
    promotion_triples.append(f'{entity_uri} <{ATLAS_NS}confidence> "{er_result["confidence"]}"^^<{XSD_NS}decimal> .')
    promotion_triples.append(f'{entity_uri} <{ATLAS_NS}matchingMethod> "{er_result["matching_method"]}"^^<{XSD_NS}string> .')
    promoted_count += 1

print(f'Entities to promote: {promoted_count}')
print(f'Triples generated:   {len(promotion_triples)}')
print()

# Write to SLGD if reachable
if neptune_available:
    print('Writing to SLGD...')
    batch_size = 50
    written = 0
    for i in range(0, len(promotion_triples), batch_size):
        batch = promotion_triples[i:i+batch_size]
        if sparql_update_slgd('INSERT DATA {\n' + '\n'.join(batch) + '\n}'):
            written += len(batch)
    print(f'  Written: {written} triples')
else:
    print('[SKIP] Neptune not reachable. Triples generated but not written.')
    print('       Run from SageMaker (inside VPC) for full promotion.')

print(f'\nPromotion complete. Run ID: {promotion_run_id}')

In [ ]:
# Generate the promotion log
print('Promotion Log')
print('=' * 60)
print(f'Run ID:              {promotion_run_id}')
print(f'Timestamp:           {promotion_timestamp}')
print(f'Operator:            workshop-participant')
print(f'Entities promoted:   {promoted_count}')
print(f'Triples generated:   {len(promotion_triples)}')
print()
print('Matching method breakdown:')
print(f'  Rule-based: {len(rule_based)} entities (confidence: 1.0)')
print(f'  ML-based:   {len(ml_based)} entities (confidence: 0.75-0.98)')
print()
print('Confidence distribution:')
confidences = [e['confidence'] for e in er_mapping.values()]
print(f'  Min:    {min(confidences):.3f}')
print(f'  Max:    {max(confidences):.3f}')
print(f'  Mean:   {sum(confidences)/len(confidences):.3f}')
print(f'  >= 0.9: {sum(1 for c in confidences if c >= 0.9)} entities')
print(f'  < 0.9:  {sum(1 for c in confidences if c < 0.9)} entities')

## Module 5 Validation Gate

The gate checks:
1. Every promoted entity has `atlas:promotedFrom` (provenance to LGD source)
2. Every promoted entity has `atlas:promotedBy` (link to promotion activity)
3. Every promoted entity has `atlas:confidence` (match confidence score)
4. The promotion activity has a timestamp and operator
5. At least 200 entities were promoted

In [ ]:
print('=' * 60)
print('MODULE 5 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Every promoted entity has promotedFrom
entities_with_provenance = sum(1 for t in promotion_triples if 'promotedFrom' in t)
if entities_with_provenance >= 200:
    print(f'[PASS] Gate 1 - {entities_with_provenance} entities have promotedFrom (provenance)')
else:
    print(f'[FAIL] Gate 1 - Only {entities_with_provenance} entities have promotedFrom')
    gate_pass = False

# Gate 2: Every promoted entity has promotedBy
entities_with_activity = sum(1 for t in promotion_triples if 'promotedBy' in t)
if entities_with_activity >= 200:
    print(f'[PASS] Gate 2 - {entities_with_activity} entities have promotedBy (activity link)')
else:
    print(f'[FAIL] Gate 2 - Only {entities_with_activity} entities have promotedBy')
    gate_pass = False

# Gate 3: Every promoted entity has confidence
entities_with_confidence = sum(1 for t in promotion_triples if 'confidence' in t and ('xsd:decimal' in t or 'XMLSchema#decimal' in t))
if entities_with_confidence >= 200:
    print(f'[PASS] Gate 3 - {entities_with_confidence} entities have confidence score')
else:
    print(f'[FAIL] Gate 3 - Only {entities_with_confidence} entities have confidence')
    gate_pass = False

# Gate 4: Promotion activity has timestamp and operator
has_timestamp = any('promotionTimestamp' in t for t in promotion_triples)
has_operator = any('promotionOperator' in t for t in promotion_triples)
if has_timestamp and has_operator:
    print(f'[PASS] Gate 4 - Promotion activity has timestamp and operator')
else:
    print(f'[FAIL] Gate 4 - Missing timestamp ({has_timestamp}) or operator ({has_operator})')
    gate_pass = False

# Gate 5: At least 200 entities promoted
if promoted_count >= 200:
    print(f'[PASS] Gate 5 - {promoted_count} entities promoted (expected >= 200)')
else:
    print(f'[FAIL] Gate 5 - Only {promoted_count} entities promoted')
    gate_pass = False

print()
if gate_pass:
    print('MODULE 5 VALIDATION: PASS')
    print('You may proceed to Module 6.')
else:
    print('MODULE 5 VALIDATION: FAIL')
    raise AssertionError('Module 5 validation gate failed.')

## WealthSignal Derivation: SPARQL CONSTRUCT from Promoted Data

With the SLGD now populated with promoted customer entities, we can derive
WealthSignal instances from the data the customer already has. This replaces
the old `event-stream.json` pre-loading approach.

The derivation logic uses SPARQL CONSTRUCT queries that pattern-match against
promoted data. Each query implements one signal-type rule:

| Signal Type | Rule | Inputs |
|---|---|---|
| `LargeDepositPattern` | Any single deposit ≥ $250,000 in the observation window, where the customer has no active wealth coverage | Transaction amount, coverage status |
| `HouseholdAggregationSignal` | Household combined checking+savings balance ≥ $1,000,000, where no individual member exceeds the threshold alone, and coverage is mixed (some members covered, some not) | Account balances, household membership, coverage status |

These queries are **deterministic**: the same promoted data always produces the
same signal instances. The derivation is reproducible and auditable.

In [ ]:
# WealthSignal Derivation via SPARQL CONSTRUCT
# These queries derive signal instances from promoted data.
# In production, they run as scheduled jobs against the SLGD.

from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD
from datetime import date, timedelta

ATLAS = Namespace('https://github.com/your-org/atlas/ontology#')
INST = Namespace('https://github.com/your-org/atlas/instance#')

print('WealthSignal Derivation from Promoted Data')
print('=' * 60)
print()

# Build a local graph simulating the SLGD with promoted data
# (In production, these queries run against the live SLGD)
g_slgd = Graph()
g_slgd.bind('atlas', ATLAS)
g_slgd.bind('inst', INST)

# Load promoted customers and their accounts/transactions
customers = atlas_synthetic.generate_customers(n=200)
accounts = atlas_synthetic.generate_accounts(customers)
transactions = atlas_synthetic.generate_transactions(accounts, lookback_days=90)

# Load advisory relationships to determine coverage status.
#
# About this fixture: data/synthetic/advisory-relationships.json contains 105
# pre-existing coverage assignments that predate the ATLAS workflow. Each entry
# carries the provenance stamp atlas:LegacyDataMigration (see extensions/
# prov-o-bindings.ttl) to distinguish it from coverage minted by the Module 8
# workflow. Module 8 will explain the engagement-vs-coverage distinction in
# detail; for now, this set defines which customers already have an active
# advisor and should NOT be flagged by a wealth-eligibility signal.
import json
with open('../data/synthetic/advisory-relationships.json') as f:
    advisory_rels = json.load(f)
covered_customers = {r['customer_id'] for r in advisory_rels if r['coverage_end_date'] is None}

# Add customers to the graph
for c in customers:
    curi = INST[f'customer-{c["customer_id"]}']
    g_slgd.add((curi, RDF.type, ATLAS.Customer))
    g_slgd.add((curi, ATLAS.customerId, Literal(c['customer_id'], datatype=XSD.string)))
    g_slgd.add((curi, ATLAS.memberOf, INST[f'household-{c["household_id"]}']))

# Add accounts
for a in accounts:
    auri = INST[f'account-{a["account_id"]}']
    g_slgd.add((auri, RDF.type, ATLAS.Account))
    g_slgd.add((auri, ATLAS.accountType, Literal(a['account_type'], datatype=XSD.string)))
    g_slgd.add((auri, ATLAS.balanceUSD, Literal(a['balance_usd'], datatype=XSD.decimal)))
    g_slgd.add((INST[f'customer-{a["customer_id"]}'], ATLAS.hasAccount, auri))

# Add transactions
for t in transactions:
    turi = INST[f'txn-{t["transaction_id"]}']
    g_slgd.add((turi, RDF.type, ATLAS.Transaction))
    g_slgd.add((turi, ATLAS.amountUSD, Literal(t['amount_usd'], datatype=XSD.decimal)))
    g_slgd.add((turi, ATLAS.transactionDate, Literal(t['transaction_date'], datatype=XSD.date)))
    g_slgd.add((turi, ATLAS.transactionType, Literal(t['transaction_type'], datatype=XSD.string)))
    g_slgd.add((INST[f'account-{t["account_id"]}'], ATLAS.hasTransaction, turi))

print(f'SLGD simulation loaded: {len(g_slgd)} triples')
print(f'  Customers: {len(customers)}')
print(f'  Accounts:  {len(accounts)}')
print(f'  Transactions: {len(transactions)}')
print(f'  Customers with active wealth coverage: {len(covered_customers)}')
print()

In [ ]:
# --- Signal 1: LargeDepositPattern ---
# Rule: deposit >= $250,000 AND customer has NO active wealth coverage
# This is the primary signal type in the ATLAS wealth-signal taxonomy.

LARGE_DEPOSIT_THRESHOLD = 250_000
observation_window_start = str(date.today() - timedelta(days=90))

large_deposit_construct = f"""
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
PREFIX inst:  <https://github.com/your-org/atlas/instance#>
PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>

CONSTRUCT {{
    ?signal a atlas:WealthSignal ;
            atlas:hasSignalType atlas:LargeDepositPattern ;
            atlas:signalDate ?txnDate ;
            atlas:evidencedBy ?txn .
    ?customer atlas:producesSignal ?signal .
}}
WHERE {{
    ?customer a atlas:Customer ;
              atlas:hasAccount ?account .
    ?account atlas:hasTransaction ?txn .
    ?txn atlas:amountUSD ?amount ;
         atlas:transactionDate ?txnDate ;
         atlas:transactionType "DEPOSIT"^^xsd:string .
    FILTER (?amount >= {LARGE_DEPOSIT_THRESHOLD})
    FILTER (?txnDate >= "{observation_window_start}"^^xsd:date)
    BIND(IRI(CONCAT(STR(inst:), "signal-ldp-", STRUUID())) AS ?signal)
}}
"""

print('Signal 1: LargeDepositPattern CONSTRUCT')
print('-' * 60)
print(f'Rule: DEPOSIT >= ${LARGE_DEPOSIT_THRESHOLD:,} AND no active wealth coverage')
print(f'Window: {observation_window_start} to today')
print()

# Execute the CONSTRUCT (without the coverage filter for the graph query,
# we apply coverage filtering in Python since it requires the advisory-relationships data)
result_graph = g_slgd.query(large_deposit_construct)

# Count signals derived (the CONSTRUCT returns triples, not a result set)
# We'll do the derivation manually to also apply the coverage filter
ldp_signals = []
for t in transactions:
    if (t['transaction_type'] == 'DEPOSIT' and
        t['amount_usd'] >= LARGE_DEPOSIT_THRESHOLD and
        t['transaction_date'] >= observation_window_start and
        t['customer_id'] not in covered_customers):
        ldp_signals.append({
            'customer_id': t['customer_id'],
            'amount': t['amount_usd'],
            'date': t['transaction_date'],
            'transaction_id': t['transaction_id'],
        })

print(f'LargeDepositPattern signals derived: {len(ldp_signals)}')
for sig in ldp_signals[:5]:
    cust = next((c for c in customers if c['customer_id'] == sig['customer_id']), None)
    name = f'{cust["first_name"]} {cust["last_name"]}' if cust else 'Unknown'
    print(f'  {name}: ${sig["amount"]:,.2f} on {sig["date"]}')
if len(ldp_signals) > 5:
    print(f'  ... and {len(ldp_signals) - 5} more')
print()

In [ ]:
# --- Signal 2: HouseholdAggregationSignal ---
# Rule: household combined checking+savings balance >= $1,000,000
#       AND no individual member exceeds the threshold alone
#       AND coverage is mixed (some members covered, some not)

from collections import defaultdict

HOUSEHOLD_THRESHOLD = 1_000_000

household_agg_construct = f"""
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
PREFIX inst:  <https://github.com/your-org/atlas/instance#>
PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>

CONSTRUCT {{
    ?signal a atlas:WealthSignal ;
            atlas:hasSignalType atlas:HouseholdAggregationSignal ;
            atlas:signalDate ?today .
    ?customer atlas:producesSignal ?signal .
}}
WHERE {{
    ?customer a atlas:Customer ;
              atlas:memberOf ?household .
    ?customer atlas:hasAccount ?account .
    ?account atlas:accountType ?acctType ;
             atlas:balanceUSD ?balance .
    FILTER (?acctType IN ("CHECKING"^^xsd:string, "SAVINGS"^^xsd:string))
    BIND(NOW() AS ?today)
    BIND(IRI(CONCAT(STR(inst:), "signal-has-", STRUUID())) AS ?signal)
}}
GROUP BY ?household
HAVING (SUM(?balance) >= {HOUSEHOLD_THRESHOLD})
"""

print('Signal 2: HouseholdAggregationSignal CONSTRUCT')
print('-' * 60)
print(f'Rule: household combined balance >= ${HOUSEHOLD_THRESHOLD:,}')
print(f'       AND no individual member >= threshold alone')
print(f'       AND mixed coverage (some covered, some not)')
print()

# Derive household signals manually (CONSTRUCT with GROUP BY is complex in rdflib)
# Build household -> members -> balances
hh_members = defaultdict(list)
for c in customers:
    hh_members[c['household_id']].append(c['customer_id'])

cust_balance = defaultdict(float)
for a in accounts:
    if a['account_type'] in ('CHECKING', 'SAVINGS'):
        cust_balance[a['customer_id']] += a['balance_usd']

has_signals = []
for hh_id, members in hh_members.items():
    if len(members) < 2:
        continue
    combined = sum(cust_balance.get(m, 0) for m in members)
    if combined < HOUSEHOLD_THRESHOLD:
        continue
    # No individual exceeds threshold alone
    if any(cust_balance.get(m, 0) >= HOUSEHOLD_THRESHOLD for m in members):
        continue
    # Mixed coverage: some covered, some not
    member_coverage = [m in covered_customers for m in members]
    if all(member_coverage) or not any(member_coverage):
        continue  # All covered or none covered — not mixed
    # This household qualifies
    uncovered = [m for m in members if m not in covered_customers]
    has_signals.append({
        'household_id': hh_id,
        'combined_balance': combined,
        'member_count': len(members),
        'uncovered_members': uncovered,
    })

print(f'HouseholdAggregationSignal signals derived: {len(has_signals)}')
for sig in has_signals[:5]:
    print(f'  Household {sig["household_id"][:12]}...: '
          f'${sig["combined_balance"]:,.2f} ({sig["member_count"]} members, '
          f'{len(sig["uncovered_members"])} uncovered)')
if len(has_signals) > 5:
    print(f'  ... and {len(has_signals) - 5} more')
print()

# Summary
total_signals = len(ldp_signals) + len(has_signals)
print('=' * 60)
print(f'Total WealthSignal instances derived: {total_signals}')
print(f'  LargeDepositPattern:        {len(ldp_signals)}')
print(f'  HouseholdAggregationSignal: {len(has_signals)}')
print()
print('Key point: these signals were DERIVED from promoted data,')
print('not loaded from a pre-computed file. The derivation is')
print('deterministic and reproducible given the same SLGD state.')

## Extending This to Your Data

### Choosing between rule-based and ML-based ER

| Use rule-based when... | Use ML-based when... |
|---|---|
| You have a shared key (customer ID, SSN, LEI) | No shared key exists across sources |
| Exact match is sufficient | Fuzzy matching needed (name variations, typos) |
| You need confidence 1.0 for compliance | You can accept confidence < 1.0 with human review |
| Audit trail must show deterministic logic | Audit trail can reference a model version |

### The most common ER pitfall

Using customer-supplied data (names, addresses) in matching rules without a SHACL
shape that constrains the rules to deterministic-only inputs. This silently lets
PII (Personally Identifiable Information) leak into the rule comparator, which
creates a compliance exposure if the matching logic is later audited.

The fix: define a SHACL shape (Module 6) that requires any ER rule input to be
tagged as `atlas:deterministic = true`. If someone adds a probabilistic input
(like an NLP-extracted name), the shape catches it.

### Designing promotion as a Step Function

For production, replace the notebook promotion script with an AWS Step Functions
state machine that:
1. Reads the ER mapping from S3
2. Validates each entity against SHACL shapes (Module 6)
3. Writes passing entities to the SLGD with PROV-O
4. Logs failures to a dead-letter queue for human review
5. Produces a promotion report as a versioned artifact

## What Changed

| Artifact | Location | Description |
|----------|----------|-------------|
| PROV-O bindings | `ontology/extensions/prov-o-bindings.ttl` | Promotion and ER activity classes, provenance properties |
| Promotion script | This notebook (cell 6) | Generates PROV-O-attributed triples for each resolved entity |
| Promotion log | This notebook (cell 7) | Entity counts, matching methods, confidence distribution |
| Populated SLGD | Neptune `atlas-slgd` cluster | 200 resolved Customer entities with full provenance |

**Key architectural point established:**

The SLGD now contains data that has been through a governed promotion path. Every
entity has provenance: where it came from (LGD), how it was matched (rule-based or
ML), with what confidence, when, and by whom. This is the audit posture MRM requires.

**What Module 6 builds on this:**

Module 6 takes the promoted data and asks: how do we mechanically enforce that
probabilistic outputs never flow into compliance-bound paths without explanation?
The answer is SHACL shapes — machine-checkable rules that validate the boundary
between deterministic and probabilistic components.